In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans

import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

In [ ]:
PATH = "/home/can/zero-to-ai-architect/runsight/endomondoHR_speed.csv"

df = pd.read_csv(PATH) 

# K-Means ile Antrenman Türü Kümeleme Denemesi

Bu notebook'ta amaç, Endomondo veri setindeki koşu antrenmanlarını (easy run, tempo run, long run, interval vb.) **K-Means** kümeleme algoritması ile otomatik olarak ayırt edebilmek.

Aşağıda sırasıyla:
1. Ham sensör verisinden (zaman, hız, nabız, konum) özellik çıkarımı yapılacak,
2. Bu özellikler K-Means'e verilerek kümeleme denenecek,
3. Sonuçlar yorumlanacak ve neden işe yaramadığı / yaramaya başladığı tartışılacak,
4. Veri setindeki gürültü (bisiklet antrenmanlarının koşu olarak etiketlenmesi gibi) tespit edilip temizlenecek.

Bu, doğrusal bir "başarı hikayesi" değil; adım adım hipotez kurup test ettiğimiz bir **deneme-yanılma süreci**. Her denemenin sonunda neyin işe yaramadığını ve bir sonraki denemeye nasıl yön verdiğini göreceksiniz.

In [ ]:
num_cols = ['longitude', 'altitude', 'latitude', 'heart_rate', 'timestamp', 'speed']

In [ ]:
def parse_arr(s):
    if isinstance(s, np.ndarray):
        return s.astype(np.float64)
    cleaned = s.replace("[", "").replace("]", "").replace(",", " ")
    return np.fromstring(cleaned, sep=" ", dtype=np.float64)


In [ ]:
for col in num_cols:
    df[col] = df[col].apply(parse_arr)

## 1. Adım: Minimal Özellik Çıkarımı

K-Means'e vereceğimiz değişkenleri hazırlıyoruz. Amaç önce en temel koşu özellikleriyle basit bir deneme yapıp kümelemenin genel olarak mümkün olup olmadığını görmek.

`extract_minimal_features` fonksiyonu her antrenman için bir sözlük döndürür. Bu sözlükte şu dört değer tutulur:

- **duration_min**: Antrenmanın dakika cinsinden süresi
- **mean_speed**: Ortalama hız
- **mean_hr**: Ortalama nabız
- **std_speed**: Hızın standart sapması (antrenman boyunca hız ne kadar değişken?)

In [ ]:
def extract_minimal_features(row):
    ts = row['timestamp']
    spd = row['speed']
    hr = row['heart_rate']

    return {
        'duration_min': ((ts[-1] - ts[0])/60.0),
        'mean_speed': np.mean(spd),
        'mean_hr': np.mean(hr),
        'std_speed': np.std(spd)
    }

## 2. Adım: K-Means - 1. Deneme

Elde edilen minimal özellikler `X` DataFrame'ine atandı. K-Means aykırı değerlere (outlier) hassas olduğu için veriler **RobustScaler** ile ölçeklendi (medyan ve çeyrekler açıklığı kullanır, ortalama/standart sapma yerine — böylece birkaç aşırı uç değer tüm ölçeklendirmeyi bozmaz).

Hedeflenen koşu sınıflarına (easy run, tempo, long run, interval gibi) uygun olacak şekilde **4 küme** oluşturulması uygun görüldü. K-Means'in sağlıklı çalışabilmesi için ham zaman serisi değil, yukarıda çıkardığımız skaler (sayısal, özet) özellikler verildi.

### Çıktının Yorumu

Çıktı bize önemli bir gerçeği gösterdi: bazı özellikler belirli bir koşu türünün baskın olduğu grupları kabaca ayırt etmemizi sağlasa da, sınıflar birbirinden **net şekilde ayrılamıyor**.

- **duration_min (koşu süresi)** ilk bakışta öne çıkan özellikti ve long run sınıfı için kısmen işe yarıyor gibi göründü. Ama bir antrenör gözünden bakıldığında, süresi 90 dakika olan her antrenman "long run" sayılamaz. Örneğin ortalama hızı 22 km/h ve ortalama nabzı 136 olan bir antrenman aslında bir **bisiklet antrenmanı** olabilir — koşu değil.
- Bu da bize başka bir sorunun varlığını gösteriyor: veri setinde `sport == 'run'` filtresiyle ayırmak bize sadece gerçek koşuları değil, yanlış etiketlenmiş kayıtları da (bisiklet gibi) beraberinde getirmiş.
- Ortalama nabzı 149 bpm olan 57 dakikalık bir koşu bize bir **tempo koşusuna** işaret edebilir.
- Hızın varyansının (std_speed) yüksek olduğu koşular ise **interval antrenmanlarını** gösteren en önemli parametre olabilirdi. Ama burada büyük bir tuzak var: yaş bilgisi olmadığı için her koşucunun kendi kapasitesine göre nabız/hız bölgelerini (zone) bilemiyoruz. Bu da bizi yanlış sınıflandırmaya sürükleyebilir.

In [ ]:
print("4 temel özellik çıkarılıyor...")
X = pd.DataFrame([extract_minimal_features(row) for _, row in df.iterrows()])

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

cluster_summary = X.copy()
cluster_summary['cluster'] = df['cluster']
print("\n--- 4 Kümenin Ortalama Değerleri ---")
print(cluster_summary.groupby('cluster').mean().round(2))

### Sonuç: 1. Deneme Başarısız

Görüldüğü gibi bu ilk kümeleme denemesi başarısız oldu. Sınıflar yeterince iyi ayrılamadı — kümeler birbirinden net bir şekilde ayrışmak yerine büyük ölçüde iç içe geçti.

In [ ]:
xs = X_scaled[:,0]
ys = X_scaled[:,1]

plt.figure(figsize=(10, 6))
plt.scatter(xs, ys, c=df['cluster'], cmap='viridis')

centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='black', s=200, alpha=0.5)

plt.show()

## 3. Adım: Yeni Özellikler ve PCA ile Boyut Küçültme

K-Means'in daha iyi kümeleyebilmesi için yeni özellikler ekliyoruz:

- **pca_route_magnitude**: `longitude`, `latitude` ve `altitude` (konum verileri) PCA ile tek boyuta indirgenerek, rotanın ne kadar "büyük" veya "yayılmış" olduğunu özetleyen tek bir sayı elde edildi. Amaç, örneğin uzun mesafeli bir rotayı yerinde sayan/kısa bir rotadan ayırt edebilmek.
- **training_effect**: Yüksek tempolu (anaerobik) ve düşük tempolu (aerobik) antrenmanları daha kolay ayırt edebilmek için eklendi. Nabzın hıza oranını (mean_hr / mean_speed) temsil eder.
- **duration**: Her antrenman 500 parçaya (data point) bölünmüş şekilde kayıtlı olduğu için, bu veriyi verimli kullanabilmek amacıyla zaman farkı 500'e bölünerek toplam antrenman süresi hesaplandı.

In [ ]:
def extract_features(row):
    ts = row['timestamp']
    spd = row['speed']
    hr = row['heart_rate']
    lon = row['longitude']
    lat = row['latitude']
    alt = row['altitude']

    dt = (ts[-1] - ts[0]) / 60


    spd = np.clip(spd, 6.0, 25.0)

    lat_rad = np.radians(lat[0])
    dx = (lon - lon[0]) * (np.cos(lat_rad) * 111139.0)
    dy = (lat - lat[0]) * 111139.0
    dz = alt - alt[0]

    coords_3d = np.column_stack((dx, dy, dz))
    pca = PCA(n_components=1)
    pca_result = pca.fit_transform(coords_3d)
    route_magnitude = float(np.std(pca_result))


    mean_speed = float(np.mean(spd))
    mean_hr = float(np.mean(hr))
    training_effect = (mean_hr / mean_speed) if mean_speed > 0.5 else 0.0

    return {
        'duration': dt,
        'mean_speed': np.mean(spd),
        'mean_hr': np.mean(hr),
        'std_speed': np.std(spd),
        'pca_route_magnitude': route_magnitude,
        'training_effect': training_effect
    }

## 4. Adım: K-Means - 2. Deneme

Bu sefer 3 sınıflı bir yaklaşım denendi. Asıl amaç easy run, long run ve anaerobik/yüksek tempolu antrenmanları ayırmaktı. Fakat çıktılar bu yaklaşımın da yetersiz olduğunu gösteriyor.

Belirli bir "uzun koşu" grubu oluştu, ancak hızın varyansı (std_speed) burada `pca_route_magnitude` değişkeninin baskınlığı altında adeta ezildi ve kümelemedeki etkisini kaybetti — yani tek bir özellik diğerlerini gölgeledi.

In [ ]:
print("3 temel özellik çıkarılıyor...")
X = pd.DataFrame([extract_features(row) for _, row in df.iterrows()])

scaler = RobustScaler(quantile_range=(5, 95))
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

cluster_summary = X.copy()
cluster_summary['cluster'] = df['cluster']
print("\n--- 3 Kümenin Ortalama Değerleri ---")
print(cluster_summary.groupby('cluster').mean().round(2))

### Sonuç: 2. Deneme de Yetersiz

Bu yaklaşım aslında daha da kötü bir kümelenmeye sebep oldu. "Simple is the best" prensibi burada hâlâ geçerli, ama K-Means tek başına bizim asıl sorunumuzu (antrenman türünü doğru sınıflandırmak) çözmekten hâlâ uzak.

In [ ]:
xs = X_scaled[:,0]
ys = X_scaled[:,1]

plt.figure(figsize=(10, 6))
plt.scatter(xs, ys, c=df['cluster'], cmap='viridis')

centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='black', s=200, alpha=0.5)

plt.show()

## 5. Adım: Bisiklet Antrenmanlarını Tespit Etme

Önceki denemelerdeki sorunun bir kısmının, veri setinde yanlış etiketlenmiş **bisiklet antrenmanlarından** kaynaklanan gürültü olabileceğini fark ettik. Bu adımda, "koşu" olarak etiketlenmiş ama aslında bisiklet olan kayıtları basit kurallarla tespit edip temizlemeyi amaçlıyoruz.

In [ ]:

def is_likely_cycling(row):
    # Kural 1: Hız çok yüksek ama HR düşük (aerobik verimlilik farkı)
    if np.mean(row['speed']) > 22 and np.mean(row['heart_rate']) < 136:
        return True

    if np.max(row['speed']) > 30:
        return True

    return False

df['activity_type'] = df.apply(lambda x: 'cycling' if is_likely_cycling(x) else 'running', axis=1)

### Tespit Sonucu

Kötü haber: şüphelerimiz doğru çıktı. Yaklaşık **822 bisiklet antrenmanı**, veri setinde koşu olarak etiketlenmiş. Bu, önceki kümeleme denemelerindeki gürültünün önemli bir kısmını açıklıyor.

In [ ]:
df['activity_type'].value_counts()


In [ ]:
df_runs = df[df['activity_type'] == 'running'].copy().reset_index(drop=True)

## 6. Adım: K-Means - 3. Deneme (Temizlenmiş Veri ile)

Son bir umutla, bisiklet antrenmanları temizlenmiş `df_runs` veri seti üzerinde tekrar K-Means deneniyor.

**Not:** Bu denemenin sonuçları kaydedilmedi (yani `df_runs` üzerine kalıcı bir değişiklik yapılmadı), çünkü bu yaklaşımın veri setini dengesizleştirme riski taşıdığı görüldü.

In [ ]:
print("3 temel özellik çıkarılıyor...")
X = pd.DataFrame([extract_features(row) for _, row in df_runs.iterrows()])

scaler = RobustScaler(quantile_range=(5, 95))
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_runs['cluster'] = kmeans.fit_predict(X_scaled)

cluster_summary = X.copy()
cluster_summary['cluster'] = df_runs['cluster']
print("\n--- 3 Kümenin Ortalama Değerleri ---")
print(cluster_summary.groupby('cluster').mean().round(2))

### Sonuç: Kısmi İlerleme, Ama Yeterli Değil

Tablonun üst kısmındaki gürültünün bir kısmı gerçekten temizlenmiş gibi görünüyor. Ancak veriyi antrenman türüne göre **tam olarak** ayrıştırmak hâlâ mümkün değil.

Belirli bir yönde kümeleme başarılı oldu, fakat bu ayrım bizim istediğimiz gibi **antrenman türüne** göre değil, sadece **antrenman uzunluğuna** göre gerçekleşmiş görünüyor. Bu yöntem, sağlıklı ve anlamlı bir özellik eklememizi sağlamaktan hâlâ çok uzak.

In [ ]:
xs = X_scaled[:,0]
ys = X_scaled[:,1]

plt.figure(figsize=(10, 6))
plt.scatter(xs, ys, c=df_runs['cluster'], cmap='viridis')

centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='black', s=200, alpha=0.5)

plt.show()

## Genel Değerlendirme

Bu notebook boyunca K-Means ile üç farklı özellik seti denendi ve üçü de antrenman türünü (easy run, tempo, long run, interval) güvenilir şekilde ayırt edemedi. Çıkardığımız dersler:

1. **Özellik seçimi kümeleme başarısını domine eder.** Ölçek farkı büyük olan bir özellik (örn. `pca_route_magnitude`), ölçeklendirme sonrası bile diğer anlamlı özellikleri (örn. `std_speed`) gölgeleyebilir.
2. **Veri kalitesi, algoritma seçiminden önce gelir.** `sport == 'run'` filtresi tek başına yeterli değildi; yaklaşık 822 bisiklet antrenmanı koşu olarak etiketlenmiş bulundu ve bu gürültü sonuçları bozuyordu.
3. **Bağlamsal bilgi eksikliği (yaş, kapasite, zone aralıkları) sınıflandırmayı zorlaştırıyor.** Aynı nabız/hız değerleri, farklı koşucular için farklı antrenman türlerine işaret edebilir.
4. **K-Means, denetimsiz (unsupervised) bir yöntem olarak burada yetersiz kaldı.** Sonraki adımda etiketli veri veya daha güçlü domain-bilgisi içeren kural tabanlı/denetimli yaklaşımlar denenmeli.